# PaliGemma Handwriting Recognition Training
## Based on: "Representing Online Handwriting for Recognition in Large Vision-Language Models"

This notebook trains PaliGemma with LoRA on the MathWriting dataset using the paper's hyperparameters.

In [ ]:
# Check GPU
!nvidia-smi

## 1. Mount Google Drive and Setup Paths

In [ ]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Setup paths
DRIVE_DATA_DIR = '/content/drive/MyDrive/mathwriting-2024'
LOCAL_DATA_DIR = '/content/mathwriting-2024'
CHECKPOINT_DIR = '/content/drive/MyDrive/math_lora_checkpoints'

## 2. Download and Cache Dataset

Downloads MathWriting dataset and caches to Drive. On subsequent runs, copies from Drive to local disk for faster access.

In [ ]:
import shutil

# Check if dataset exists in Drive (cached)
if os.path.exists(DRIVE_DATA_DIR):
    print(f"✓ Dataset found in Drive: {DRIVE_DATA_DIR}")
    print("Copying to local disk for faster training...")
    !cp -r {DRIVE_DATA_DIR} {LOCAL_DATA_DIR}
    print(f"✓ Dataset copied to {LOCAL_DATA_DIR}")
else:
    print("Downloading MathWriting dataset...")
    !wget -O /content/mathwriting-2024.tgz https://storage.googleapis.com/mathwriting_data/mathwriting-2024.tgz
    
    print("Extracting dataset...")
    !tar -xzf /content/mathwriting-2024.tgz -C /content/
    
    print("Caching to Drive for future runs...")
    !cp -r {LOCAL_DATA_DIR} {DRIVE_DATA_DIR}
    print("✓ Dataset cached to Drive")

# Verify dataset structure
!ls -lh {LOCAL_DATA_DIR}

## 3. Clone GitHub Repository

In [ ]:
# Clone the repo with training scripts
!git clone https://github.com/hudsonmp/realtime-math.git
%cd realtime-math

## 4. Install Dependencies

In [ ]:
!pip install -q transformers peft accelerate huggingface-hub pillow torch torchvision tqdm

## 5. Configure HuggingFace Authentication

**Important:** Add your HuggingFace token to Colab Secrets:
1. Click the 🔑 icon in the left sidebar
2. Add a secret named `HF_TOKEN`
3. Paste your HuggingFace token (get it from https://huggingface.co/settings/tokens)
4. Make sure you've accepted the PaliGemma license at https://huggingface.co/google/paligemma-3b-pt-224

In [ ]:
from google.colab import userdata
import os

try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = HF_TOKEN
    print("✓ HuggingFace token loaded from Colab secrets")
except:
    print("⚠️  HF_TOKEN not found in Colab secrets!")
    print("Please add it using the 🔑 icon in the sidebar")

## 6. Train Model

### Hyperparameters from Paper:
- **Model:** PaliGemma-3B (google/paligemma-3b-pt-224)
- **LoRA:** r=16, alpha=32, dropout=0.05, targets=[q_proj, v_proj]
- **Batch size:** 4 (effective 16 with grad_accum=4)
- **Learning rate:** 1e-4
- **Epochs:** 10
- **Optimizer:** AdamW
- **Scheduler:** Linear with 100 warmup steps
- **Image size:** 224x224 (2 lines, time+distance rendering)
- **Sequence:** 16 points/stroke, relative coordinates, [0,224] normalization

In [ ]:
# Import and run training
from train import MathTrainer

# Initialize trainer with paper hyperparameters
trainer = MathTrainer(
    data_dir=LOCAL_DATA_DIR,
    model_name="google/paligemma-3b-pt-224",
    output_dir=CHECKPOINT_DIR,
    device="cuda",
    use_gradient_checkpointing=True  # Memory efficient for A100
)

# Train with paper hyperparameters
trainer.train(
    epochs=10,
    batch_size=4,
    lr=1e-4,
    grad_accum=4  # Effective batch size = 16
)

## 7. Export Model (Optional)

Merge LoRA weights and optionally quantize for deployment.

In [ ]:
from export_model import ModelExporter

# Find the best checkpoint
import glob
best_checkpoint = sorted(glob.glob(f"{CHECKPOINT_DIR}/best_lora*"))[-1]
print(f"Using checkpoint: {best_checkpoint}")

# Export merged model
exporter = ModelExporter(
    base_model="google/paligemma-3b-pt-224",
    lora_path=best_checkpoint
)

# Merge and save
merged_model = exporter.merge_and_save(
    output_dir="/content/drive/MyDrive/merged_model"
)

print("✓ Model exported and saved to Drive")

## 8. Test Inference (Optional)

In [ ]:
# Quick test with a sample from the validation set
from data_preprocessing import MathWritingDataset

# Load validation dataset
val_dataset = MathWritingDataset(LOCAL_DATA_DIR, split='valid')
sample = val_dataset[0]

print(f"Ground Truth: {sample['label']}")
print(f"Stroke Text (first 100 chars): {sample['stroke_text'][:100]}...")

# Display the rendered image
from IPython.display import display
display(sample['image'])

# Test inference (if model is loaded)
# prediction = trainer.processor.decode(
#     trainer.model.generate(**inputs, max_length=64)[0],
#     skip_special_tokens=True
# )
# print(f"Prediction: {prediction}")

## 9. Monitor Training (Optional)

View recent checkpoints and training progress.

In [ ]:
# List all checkpoints
!ls -lh {CHECKPOINT_DIR}

# Show best checkpoint info
import glob
best_checkpoints = glob.glob(f"{CHECKPOINT_DIR}/best_lora*")
if best_checkpoints:
    print("\nBest checkpoints:")
    for ckpt in best_checkpoints:
        print(f"  {ckpt}")